
# Selecting Hyperparameters


In [3]:
# from transformers import AutoTokenizer
# import json

# tokenizer = AutoTokenizer.from_pretrained(
#     "Qwen/Qwen2.5-3B-Instruct"
# )

# lengths = []

# with open("stage2_train.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         item = json.loads(line)

#         text = (
#             item["messages"][0]["content"]
#             + "\n"
#             + item["messages"][1]["content"]
#         )

#         lengths.append(len(tokenizer.encode(text)))

# print("avg:", sum(lengths)/len(lengths))
# print("max:", max(lengths))

In [4]:
# selecting Hyperparameters
# max_seq_length = 4096

# r = 16
# lora_alpha = 32

# per_device_train_batch_size = 2
# gradient_accumulation_steps = 4

# learning_rate = 2e-4

# num_train_epochs = 3

# load_in_4bit = True

## step 1

In [5]:
# Step 1: Install correct Unsloth and dependencies for Google Colab
# !pip install --upgrade pip
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers triton bitsandbytes

In [6]:
# r = 16
# lora_alpha = 32
# lora_dropout = 0.05

##Load Model

In [7]:
from unsloth import FastLanguageModel

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1568: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

##Add LoRA

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


##Convert Dataset

Your planner dataset should become:

In [9]:
from datasets import Dataset, load_dataset
dataset = load_dataset(
    "json",
    data_files="stage2_train.jsonl",
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

##Training

In [10]:
from huggingface_hub import login

login()

In [11]:
import trl
import transformers

print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)

TRL: 0.29.1
Transformers: 5.5.0


In [12]:
def format_example(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

dataset = dataset.map(format_example)

Map:   0%|          | 0/4571 [00:00<?, ? examples/s]

In [17]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=50,
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        output_dir="outputs",
        report_to="none",
        # save_strategy="epoch",
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
    ),
)

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4571 [00:00<?, ? examples/s]

In [18]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,571 | Num Epochs = 2 | Total steps = 1,144
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.104503
2,1.013149
3,1.002491
4,0.988951
5,1.020850
6,1.165092
7,1.186047
8,0.957744
9,0.905383
10,0.862789


Step,Training Loss
1,1.104503
2,1.013149
3,1.002491
4,0.988951
5,1.020850
6,1.165092
7,1.186047
8,0.957744
9,0.905383
10,0.862789


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1144/tokenizer_config.json.


TrainOutput(global_step=1144, training_loss=0.2464641004253726, metrics={'train_runtime': 12360.3199, 'train_samples_per_second': 0.74, 'train_steps_per_second': 0.093, 'total_flos': 1.528419536790651e+17, 'train_loss': 0.2464641004253726})

##Save Adapter

In [19]:
model.save_pretrained("workflow_planner_lora")
tokenizer.save_pretrained("workflow_planner_lora")

Unsloth: Restored added_tokens_decoder metadata in workflow_planner_lora/tokenizer_config.json.


('workflow_planner_lora/tokenizer_config.json',
 'workflow_planner_lora/chat_template.jinja',
 'workflow_planner_lora/tokenizer.json')

save in hugging face

In [21]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '68cc1c854a8d1279cd0385fe', 'name': 'Karthik1338', 'fullname': 'Karthik P R', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/SWIZTipoI-esJTg530imr.png', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-20T17:20:03.000Z'}}


In [22]:
from huggingface_hub import logout, login

logout()
login()

In [24]:
model.push_to_hub("karthik1338/workflow-stage2-lora")
tokenizer.push_to_hub("karthik1338/workflow-stage2-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  120MB            

Saved model to https://huggingface.co/karthik1338/workflow-stage2-lora


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpkyg2xb56/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkyg2xb56/tokenizer.json:  30%|###       | 3.46MB / 11.4MB            

# Next Step: Run Inference

Load the adapter and test it.

In [25]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model.load_adapter("workflow_planner_lora")

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=256, bias=True)
            (lora_dropout): Mod

In [29]:
def generate_json(prompt):
  messages = [
      {
          "role": "user",
          "content": prompt
      }
  ]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  )

  inputs = tokenizer(
      text,
      return_tensors="pt"
  ).to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=2048,
      temperature=0.1,
  )

  print(
      tokenizer.decode(
          outputs[0],
          skip_special_tokens=True
      )
  )

In [27]:
prompt1 = """
User Query:
Download webpage, extract links, filter PDFs, save results

Available APIs:
downloadurl
getlinkfromhtml
filterfiles
savefile
"""

generate_json(prompt1)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Download webpage, extract links, filter PDFs, save results

Available APIs:
downloadurl
getlinkfromhtml
filterfiles
savefile

assistant
{"steps": [{"id": 1, "tool": "downloadurl", "inputs": [], "depends_on": [], "output": "webpage_download_url"}, {"id": 2, "tool": "getlinkfromhtml", "inputs": ["webpage_download_url"], "depends_on": [1], "output": "links_from_html"}, {"id": 3, "tool": "filterfiles", "inputs": ["links_from_html"], "depends_on": [2], "output": "filtered_pdf_files"}, {"id": 4, "tool": "savefile", "inputs": ["filtered_pdf_files"], "depends_on": [3], "output": "saved_file"}]}


In [28]:
prompt2 ='''
User Query:
Download webpage,
extract links,
extract images,
save both.
'''
generate_json(prompt2)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Download webpage,
extract links,
extract images,
save both.

assistant
{"steps": [{"id": 1, "tool": "is_workflow_actions_downloadwebpage", "inputs": [], "depends_on": [], "output": "downloaded_webpage"}, {"id": 2, "tool": "is_workflow_actions_extractlinks", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_links"}, {"id": 3, "tool": "is_workflow_actions_extractimages", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_images"}]}


In [30]:
prompt2 ='''
User Query:
Download webpage,
extract links,
extract images,
save both.
'''
generate_json(prompt2)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Download webpage,
extract links,
extract images,
save both.

assistant
{"steps": [{"id": 1, "tool": "is_workflow_actions_downloadwebpage", "inputs": [], "depends_on": [], "output": "downloaded_webpage"}, {"id": 2, "tool": "is_workflow_actions_extractlinks", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_links"}, {"id": 3, "tool": "is_workflow_actions_extractimages", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_images"}]}


In [31]:
prompt3 = '''
Create a workflow that downloads a PDF from a URL, extracts all text from the PDF, summarizes the extracted text, and saves the summary to a file.'''
generate_json(prompt3)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Create a workflow that downloads a PDF from a URL, extracts all text from the PDF, summarizes the extracted text, and saves the summary to a file.
assistant
{"steps": [{"id": 1, "tool": "com_alexhay_ToolboxProForShortcuts_GetPDFTextIntent", "inputs": ["pdf_url"], "depends_on": [], "output": "pdf_text"}, {"id": 2, "tool": "com_alexhay_ToolboxProForShortcuts_TextSummarizationIntent", "inputs": ["pdf_text"], "depends_on": [1], "output": "text_summary"}, {"id": 3, "tool": "com_alexhay_ToolboxProForShortcuts_SaveFileIntent", "inputs": ["text_summary"], "depends_on": [2], "output": "saved_file"}]}


In [32]:
prompt4='''
Download a webpage, extract all links and images from it, combine the extracted information into a report, and save the report.'''
generate_json(prompt4)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Download a webpage, extract all links and images from it, combine the extracted information into a report, and save the report.
assistant
{"steps": [{"id": 1, "tool": "is_workflow_actions_downloadwebpage", "inputs": [], "depends_on": [], "output": "downloaded_webpage"}, {"id": 2, "tool": "is_workflow_actions_extractlinks", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_links"}, {"id": 3, "tool": "is_workflow_actions_extractimages", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_images"}, {"id": 4, "tool": "com_alexhay_ToolboxProForShortcuts_CreateReportIntent", "inputs": ["report_title", "report_content"], "depends_on": [], "output": "created_report"}]}


In [33]:
prompt5='''Download a webpage, extract all links and images, create a report using both extracted links and images, save the report, and email the saved report.'''
generate_json(prompt5)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Download a webpage, extract all links and images, create a report using both extracted links and images, save the report, and email the saved report.
assistant
{"steps": [{"id": 1, "tool": "is_workflow_actions_downloadwebpage", "inputs": [], "depends_on": [], "output": "downloaded_webpage"}, {"id": 2, "tool": "is_workflow_actions_extractlinks", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_links"}, {"id": 3, "tool": "is_workflow_actions_extractimages", "inputs": ["downloaded_webpage"], "depends_on": [1], "output": "extracted_images"}, {"id": 4, "tool": "com_apple_mobilenotes_SharingExtension", "inputs": ["report_content"], "depends_on": [], "output": "create_note"}, {"id": 5, "tool": "com_apple_mobilenotes_SharingExtension", "inputs": ["report_content"], "depends_on": [], "output": "create_note"}]}
